# LAB 12 — BÀI KIỂM TRA TỔNG HỢP NLP SAU KHÓA HỌC

**Nền tảng:** Google Colab — GPU T4 miễn phí  
**Phạm vi ôn tập:** phân loại văn bản truyền thống, Transformer/BERT, tóm tắt encoder–decoder, LoRA/QLoRA  
**Hình thức:** cá nhân — nộp notebook đã chạy và giữ nguyên output  
**Thang điểm:** 100

> Trước khi chạy: **Runtime → Change runtime type → T4 GPU**.  
> Chạy tuần tự từ trên xuống. Không giữ đồng thời nhiều mô hình trên GPU.

## Bối cảnh

Bạn xây dựng một hệ thống NLP thu nhỏ gồm ba mô-đun:

1. Phân loại cảm xúc phản hồi sinh viên bằng baseline TF–IDF và PhoBERT.
2. Tóm tắt hội thoại bằng T5.
3. Tinh chỉnh một LLM tiếng Việt bằng QLoRA.

Notebook đã cung cấp hạ tầng và cấu hình an toàn cho T4. Các vị trí cần sinh viên hoàn thiện hoặc giải thích được đánh dấu **`TODO [SV]`**.

## Cơ cấu điểm

| Phần | Nội dung | Điểm |
|---|---|---:|
| A | Môi trường, GPU, tái lập thí nghiệm | 5 |
| B | EDA và baseline TF–IDF + Logistic Regression | 20 |
| C | Fine-tune PhoBERT và phân tích lỗi | 30 |
| D | Fine-tune T5 cho tóm tắt và đánh giá ROUGE | 20 |
| E | QLoRA cho LLM và so sánh trước/sau | 20 |
| F | Báo cáo, kết luận, khả năng tái lập | 5 |
| **Tổng** |  | **100** |

### Quy tắc nộp bài

- Điền thông tin sinh viên.
- Không xóa output, biểu đồ, bảng đánh giá.
- Mỗi phần phải có nhận xét bằng lời, không chỉ có metric.
- Giữ `QUICK_MODE=True` khi dùng T4 miễn phí.
- Nộp notebook `.ipynb` và file `lab12_final_report.json` được tạo ở cuối bài.

## Thông tin sinh viên

- **Họ và tên:** ...
- **Mã sinh viên:** ...
- **Lớp:** ...
- **Ngày thực hiện:** ...

## PHẦN A — CHUẨN BỊ MÔI TRƯỜNG (5 điểm)

In [ ]:
# Cài thư viện trước mọi import Hugging Face.
# Không cài lại PyTorch để giữ bản CUDA tương thích sẵn có của Colab.
import subprocess
import sys

PACKAGES = [
    "transformers==5.14.1",
    "datasets==5.0.1",
    "accelerate==1.14.0",
    "peft==0.19.1",
    "trl==1.8.0",
    "bitsandbytes==0.49.2",
    "evaluate>=0.4.3,<1",
    "rouge_score>=0.1.2",
    "sentencepiece>=0.2.1",
    "underthesea==9.5.0",
    "scikit-learn>=1.4,<2",
    "pandas>=2.0",
    "matplotlib>=3.8",
    "safetensors>=0.5.0",
]

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade",
    "--upgrade-strategy", "only-if-needed", *PACKAGES
])
print("Cài thư viện hoàn tất.")

In [ ]:
import gc
import inspect
import json
import math
import os
import random
import shutil
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from datasets import Dataset, DatasetDict, load_dataset
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
    f1_score,
    precision_recall_fscore_support,
)
from underthesea import word_tokenize

from transformers import (
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    DataCollatorWithPadding,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    Trainer,
    TrainingArguments,
    set_seed,
)
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer
from rouge_score import rouge_scorer

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

SEED = 42
QUICK_MODE = True
OUTPUT_ROOT = Path("/content/lab12_outputs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError(
        "Không phát hiện GPU. Chọn Runtime → Change runtime type → T4 GPU rồi chạy lại."
    )

GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
BF16_OK = torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if BF16_OK else torch.float16
USE_TF32 = torch.cuda.get_device_capability(0)[0] >= 8

print("GPU:", GPU_NAME)
print(f"VRAM: {GPU_VRAM_GB:.2f} GB")
print("Compute dtype:", COMPUTE_DTYPE)
print("QUICK_MODE:", QUICK_MODE)

assert "T4" in GPU_NAME or GPU_VRAM_GB >= 14, (
    "Cấu hình được thiết kế cho T4 hoặc GPU có ít nhất khoảng 14 GB VRAM."
)

In [ ]:
def cleanup_gpu():
    """Giải phóng tham chiếu trước khi gọi hàm này, sau đó dọn cache CUDA."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def gpu_memory():
    if not torch.cuda.is_available():
        return {}
    return {
        "allocated_gb": round(torch.cuda.memory_allocated() / 1024**3, 3),
        "reserved_gb": round(torch.cuda.memory_reserved() / 1024**3, 3),
        "max_allocated_gb": round(torch.cuda.max_memory_allocated() / 1024**3, 3),
    }


def select_subset(split, size, seed=SEED):
    split = split.shuffle(seed=seed)
    if size is None or size >= len(split):
        return split
    return split.select(range(size))


def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {
        "total": total,
        "trainable": trainable,
        "trainable_percent": 100 * trainable / max(total, 1),
    }


def trainer_tokenizer_kwargs(trainer_cls, tokenizer):
    """Tương thích Transformers dùng `tokenizer` hoặc `processing_class`."""
    params = inspect.signature(trainer_cls.__init__).parameters
    if "processing_class" in params:
        return {"processing_class": tokenizer}
    return {"tokenizer": tokenizer}


RESULTS = {}
print(gpu_memory())

### TODO [SV-A1] — Nhận xét môi trường

Viết 2–3 câu:

1. GPU được cấp là gì và có bao nhiêu VRAM?
2. Vì sao notebook dùng FP16 trên T4?
3. Vì sao cần giải phóng mô hình trước khi chuyển sang phần tiếp theo?

> **TRẢ LỜI:** ...

## PHẦN B — PHÂN LOẠI VĂN BẢN TRUYỀN THỐNG (20 điểm)

In [ ]:
def load_uit_vsfc():
    """Tải UIT-VSFC; dùng Parquet dự phòng nếu loader chính thất bại."""
    try:
        return load_dataset("uitnlp/vietnamese_students_feedback")
    except Exception as exc:
        print("Loader chính chưa thành công:", repr(exc))
        base = (
            "https://huggingface.co/datasets/uitnlp/"
            "vietnamese_students_feedback/resolve/refs%2Fconvert%2Fparquet/"
            "default"
        )
        data_files = {
            "train": f"{base}/train/0000.parquet",
            "validation": f"{base}/validation/0000.parquet",
            "test": f"{base}/test/0000.parquet",
        }
        return load_dataset("parquet", data_files=data_files)


vi_raw = load_uit_vsfc()

if QUICK_MODE:
    VI_TRAIN_SIZE, VI_VAL_SIZE, VI_TEST_SIZE = 2500, 400, 500
else:
    VI_TRAIN_SIZE, VI_VAL_SIZE, VI_TEST_SIZE = None, None, None

vi_selected = DatasetDict({
    "train": select_subset(vi_raw["train"], VI_TRAIN_SIZE, SEED),
    "validation": select_subset(vi_raw["validation"], VI_VAL_SIZE, SEED + 1),
    "test": select_subset(vi_raw["test"], VI_TEST_SIZE, SEED + 2),
})

print(vi_selected)
print("Features:", vi_raw["train"].features)
print("Mẫu:", vi_selected["train"][0])

In [ ]:
# EDA tối thiểu bắt buộc
train_df = vi_selected["train"].to_pandas()
val_df = vi_selected["validation"].to_pandas()
test_df = vi_selected["test"].to_pandas()

train_df["char_length"] = train_df["sentence"].astype(str).str.len()
train_df["word_length"] = train_df["sentence"].astype(str).str.split().str.len()

label_feature = vi_raw["train"].features.get("sentiment")
if hasattr(label_feature, "names") and label_feature.names:
    LABEL_NAMES = list(label_feature.names)
else:
    LABEL_NAMES = ["negative", "neutral", "positive"]

label_counts = train_df["sentiment"].value_counts().sort_index()
eda_table = pd.DataFrame({
    "label_id": label_counts.index,
    "label_name": [
        LABEL_NAMES[int(i)] if int(i) < len(LABEL_NAMES) else str(i)
        for i in label_counts.index
    ],
    "count": label_counts.values,
    "ratio": (label_counts.values / label_counts.sum()).round(4),
})

display(eda_table)
display(train_df[["char_length", "word_length"]].describe())

label_counts.plot(kind="bar", title="Phân bố nhãn cảm xúc — train")
plt.xlabel("Label ID")
plt.ylabel("Số mẫu")
plt.show()

train_df["word_length"].plot(
    kind="hist", bins=30, title="Phân bố độ dài phản hồi"
)
plt.xlabel("Số từ")
plt.show()

### TODO [SV-B1] — EDA (5 điểm)

- Nhãn nào nhiều nhất, nhãn nào ít nhất?
- Dữ liệu có mất cân bằng đáng kể không?
- Chọn một giá trị `max_length` hợp lý cho Transformer ở phần C và giải thích.

> **TRẢ LỜI:** ...

In [ ]:
# Baseline TF–IDF + Logistic Regression.
# TODO [SV-B2]: thay đổi ÍT NHẤT MỘT siêu tham số và ghi lý do trong ô nhận xét.
TFIDF_MAX_FEATURES = 30_000
TFIDF_NGRAM_RANGE = (1, 2)
LOGREG_C = 2.0

tfidf = TfidfVectorizer(
    lowercase=True,
    ngram_range=TFIDF_NGRAM_RANGE,
    min_df=2,
    max_df=0.98,
    max_features=TFIDF_MAX_FEATURES,
    sublinear_tf=True,
)

X_train_tfidf = tfidf.fit_transform(train_df["sentence"].astype(str))
X_val_tfidf = tfidf.transform(val_df["sentence"].astype(str))
X_test_tfidf = tfidf.transform(test_df["sentence"].astype(str))

y_train = train_df["sentiment"].to_numpy()
y_val = val_df["sentiment"].to_numpy()
y_test = test_df["sentiment"].to_numpy()

baseline_model = LogisticRegression(
    C=LOGREG_C,
    max_iter=1000,
    class_weight="balanced",
    random_state=SEED,
)

baseline_model.fit(X_train_tfidf, y_train)
val_pred = baseline_model.predict(X_val_tfidf)
test_pred = baseline_model.predict(X_test_tfidf)

baseline_metrics = {
    "val_accuracy": accuracy_score(y_val, val_pred),
    "val_f1_macro": f1_score(y_val, val_pred, average="macro"),
    "test_accuracy": accuracy_score(y_test, test_pred),
    "test_f1_macro": f1_score(y_test, test_pred, average="macro"),
}
RESULTS["baseline_tfidf"] = baseline_metrics

display(pd.DataFrame([baseline_metrics]).round(4))
print(classification_report(
    y_test,
    test_pred,
    target_names=LABEL_NAMES,
    zero_division=0,
))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_pred,
    display_labels=LABEL_NAMES,
    xticks_rotation=30,
)
plt.title("Baseline TF–IDF + Logistic Regression")
plt.show()

In [ ]:
# Hiển thị các lỗi có độ tin cậy cao để phân tích.
test_prob = baseline_model.predict_proba(X_test_tfidf)
test_conf = test_prob.max(axis=1)

baseline_errors = test_df[["sentence", "sentiment"]].copy()
baseline_errors["prediction"] = test_pred
baseline_errors["confidence"] = test_conf
baseline_errors = baseline_errors[
    baseline_errors["sentiment"] != baseline_errors["prediction"]
].sort_values("confidence", ascending=False)

display(baseline_errors.head(12))

### TODO [SV-B3] — Nhận xét baseline (15 điểm)

1. Ghi lại siêu tham số bạn đã thay đổi và lý do.
2. Báo cáo Accuracy và Macro-F1 trên test.
3. Phân tích confusion matrix: cặp nhãn nào dễ nhầm nhất?
4. Chọn 2 lỗi dự đoán và nêu nguyên nhân có thể.

> **TRẢ LỜI:** ...

## PHẦN C — FINE-TUNE PHOBERT (30 điểm)

In [ ]:
PHOBERT_MODEL_ID = "vinai/phobert-base-v2"

# TODO [SV-C1]: điều chỉnh theo EDA; 128 là mặc định an toàn cho T4.
PHOBERT_MAX_LENGTH = 128

def prepare_vi_batch(batch):
    segmented = [
        word_tokenize(str(text), format="text")
        for text in batch["sentence"]
    ]
    return {"text": segmented, "labels": batch["sentiment"]}


vi_prepared = vi_selected.map(
    prepare_vi_batch,
    batched=True,
    batch_size=256,
    desc="Tách từ tiếng Việt",
)

keep_columns = {"text", "labels"}
remove_columns = [
    c for c in vi_prepared["train"].column_names
    if c not in keep_columns
]
vi_prepared = vi_prepared.remove_columns(remove_columns)

phobert_tokenizer = AutoTokenizer.from_pretrained(
    PHOBERT_MODEL_ID,
    use_fast=True,
)

def tokenize_phobert(batch):
    return phobert_tokenizer(
        batch["text"],
        truncation=True,
        max_length=PHOBERT_MAX_LENGTH,
    )

vi_tokenized = vi_prepared.map(
    tokenize_phobert,
    batched=True,
    remove_columns=["text"],
    desc="Tokenize PhoBERT",
)

print(vi_tokenized)
print(vi_tokenized["train"][0])

In [ ]:
def compute_classification_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0,
    )
    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1,
    }


def build_classifier_args(output_dir):
    params = inspect.signature(TrainingArguments.__init__).parameters
    kwargs = dict(
        output_dir=str(output_dir),
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=1 if QUICK_MODE else 3,
        weight_decay=0.01,
        warmup_ratio=0.1,
        logging_steps=25,
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        report_to="none",
        seed=SEED,
        data_seed=SEED,
        fp16=not BF16_OK,
        bf16=BF16_OK,
        tf32=USE_TF32,
        gradient_checkpointing=True,
        optim="adamw_torch_fused",
        save_strategy="epoch",
        logging_strategy="steps",
    )
    if "eval_strategy" in params:
        kwargs["eval_strategy"] = "epoch"
    elif "evaluation_strategy" in params:
        kwargs["evaluation_strategy"] = "epoch"
    return TrainingArguments(**{
        k: v for k, v in kwargs.items() if k in params
    })


num_labels = len(set(vi_selected["train"]["sentiment"]))
id2label = {
    i: LABEL_NAMES[i] if i < len(LABEL_NAMES) else str(i)
    for i in range(num_labels)
}
label2id = {v: k for k, v in id2label.items()}

phobert_model = AutoModelForSequenceClassification.from_pretrained(
    PHOBERT_MODEL_ID,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

phobert_args = build_classifier_args(
    OUTPUT_ROOT / "phobert_sentiment"
)

phobert_trainer = Trainer(
    model=phobert_model,
    args=phobert_args,
    train_dataset=vi_tokenized["train"],
    eval_dataset=vi_tokenized["validation"],
    data_collator=DataCollatorWithPadding(phobert_tokenizer),
    compute_metrics=compute_classification_metrics,
    **trainer_tokenizer_kwargs(Trainer, phobert_tokenizer),
)

print(count_parameters(phobert_model))
phobert_train_result = phobert_trainer.train()
phobert_test_output = phobert_trainer.predict(vi_tokenized["test"])
phobert_test_metrics = {
    k.replace("test_", ""): float(v)
    for k, v in phobert_test_output.metrics.items()
    if isinstance(v, (int, float))
}
RESULTS["phobert"] = phobert_test_metrics

display(pd.DataFrame([phobert_test_metrics]).round(4))

In [ ]:
phobert_pred = np.argmax(phobert_test_output.predictions, axis=-1)
phobert_true = np.asarray(phobert_test_output.label_ids)

print(classification_report(
    phobert_true,
    phobert_pred,
    target_names=LABEL_NAMES,
    zero_division=0,
))

ConfusionMatrixDisplay.from_predictions(
    phobert_true,
    phobert_pred,
    display_labels=LABEL_NAMES,
    xticks_rotation=30,
)
plt.title("PhoBERT — test set")
plt.show()

phobert_errors = test_df[["sentence", "sentiment"]].copy()
phobert_errors["prediction"] = phobert_pred
phobert_errors = phobert_errors[
    phobert_errors["sentiment"] != phobert_errors["prediction"]
]
display(phobert_errors.head(15))

In [ ]:
comparison_cls = pd.DataFrame([
    {
        "model": "TF-IDF + Logistic Regression",
        "accuracy": RESULTS["baseline_tfidf"]["test_accuracy"],
        "f1_macro": RESULTS["baseline_tfidf"]["test_f1_macro"],
    },
    {
        "model": "PhoBERT",
        "accuracy": RESULTS["phobert"].get("accuracy", np.nan),
        "f1_macro": RESULTS["phobert"].get("f1_macro", np.nan),
    },
])
display(comparison_cls.round(4))

### TODO [SV-C2] — Phân tích Transformer (30 điểm)

1. Giải thích vì sao cần word segmentation trước PhoBERT.
2. Giải thích lựa chọn `PHOBERT_MAX_LENGTH`.
3. So sánh Macro-F1 của PhoBERT với baseline.
4. PhoBERT cải thiện nhãn nào nhiều nhất? Nhãn nào vẫn khó?
5. Chọn 3 lỗi của PhoBERT và phân nhóm nguyên nhân: câu quá ngắn, phủ định, mơ hồ, lỗi chính tả, thiếu ngữ cảnh hoặc nhãn nhiễu.
6. Đề xuất 2 cải tiến nhưng vẫn phù hợp giới hạn T4.

> **TRẢ LỜI:** ...

In [ ]:
# Giải phóng GPU trước phần tóm tắt.
del phobert_trainer, phobert_model, phobert_test_output
cleanup_gpu()
print(gpu_memory())

## PHẦN D — TÓM TẮT HỘI THOẠI VỚI T5 (20 điểm)

In [ ]:
def iterable_to_dataset(iterable, size, seed=SEED, buffer_size=10_000):
    stream = iterable.shuffle(seed=seed, buffer_size=buffer_size)
    rows = list(stream.take(size))
    if not rows:
        raise RuntimeError("Không lấy được dữ liệu từ Hugging Face Hub.")
    return Dataset.from_list(rows)


def load_streaming_subset(repo_id, split, size, seed):
    stream = load_dataset(repo_id, split=split, streaming=True)
    return iterable_to_dataset(stream, size=size, seed=seed)


if QUICK_MODE:
    SUM_TRAIN_SIZE, SUM_VAL_SIZE, SUM_TEST_SIZE = 800, 100, 120
    SUM_EPOCHS = 1
else:
    SUM_TRAIN_SIZE, SUM_VAL_SIZE, SUM_TEST_SIZE = 4000, 400, 400
    SUM_EPOCHS = 2

samsum = DatasetDict({
    "train": load_streaming_subset(
        "knkarthick/samsum", "train", SUM_TRAIN_SIZE, SEED
    ),
    "validation": load_streaming_subset(
        "knkarthick/samsum", "validation", SUM_VAL_SIZE, SEED + 1
    ),
    "test": load_streaming_subset(
        "knkarthick/samsum", "test", SUM_TEST_SIZE, SEED + 2
    ),
})

print(samsum)
display(pd.DataFrame(samsum["train"][:3]))

In [ ]:
T5_MODEL_ID = "google-t5/t5-small"
SOURCE_MAX_LENGTH = 256
TARGET_MAX_LENGTH = 72

t5_tokenizer = AutoTokenizer.from_pretrained(T5_MODEL_ID)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(T5_MODEL_ID)

def preprocess_samsum(batch):
    inputs = ["summarize: " + text for text in batch["dialogue"]]
    model_inputs = t5_tokenizer(
        inputs,
        max_length=SOURCE_MAX_LENGTH,
        truncation=True,
    )
    labels = t5_tokenizer(
        text_target=batch["summary"],
        max_length=TARGET_MAX_LENGTH,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


samsum_tokenized = samsum.map(
    preprocess_samsum,
    batched=True,
    remove_columns=samsum["train"].column_names,
    desc="Tokenize SAMSum",
)
print(samsum_tokenized)

In [ ]:
rouge = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True,
)

def compute_rouge_metrics(eval_pred):
    predictions, labels = eval_pred
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    decoded_preds = t5_tokenizer.batch_decode(
        predictions,
        skip_special_tokens=True,
    )
    labels = np.where(
        labels != -100,
        labels,
        t5_tokenizer.pad_token_id,
    )
    decoded_labels = t5_tokenizer.batch_decode(
        labels,
        skip_special_tokens=True,
    )

    scores = {"rouge1": [], "rouge2": [], "rougeL": []}
    for pred, ref in zip(decoded_preds, decoded_labels):
        item = rouge.score(ref, pred)
        for key in scores:
            scores[key].append(item[key].fmeasure)

    return {
        key: round(float(np.mean(values)), 4)
        for key, values in scores.items()
    }


def build_seq2seq_args(output_dir):
    params = inspect.signature(
        Seq2SeqTrainingArguments.__init__
    ).parameters
    kwargs = dict(
        output_dir=str(output_dir),
        learning_rate=5e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=2,
        num_train_epochs=SUM_EPOCHS,
        weight_decay=0.01,
        warmup_ratio=0.06,
        logging_steps=20,
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="rougeL",
        greater_is_better=True,
        report_to="none",
        seed=SEED,
        data_seed=SEED,
        fp16=not BF16_OK,
        bf16=BF16_OK,
        tf32=USE_TF32,
        gradient_checkpointing=True,
        optim="adamw_torch_fused",
        predict_with_generate=True,
        generation_max_length=TARGET_MAX_LENGTH,
        generation_num_beams=4,
        save_strategy="epoch",
        logging_strategy="steps",
    )
    if "eval_strategy" in params:
        kwargs["eval_strategy"] = "epoch"
    elif "evaluation_strategy" in params:
        kwargs["evaluation_strategy"] = "epoch"
    return Seq2SeqTrainingArguments(**{
        k: v for k, v in kwargs.items() if k in params
    })


t5_args = build_seq2seq_args(OUTPUT_ROOT / "t5_samsum")
t5_trainer = Seq2SeqTrainer(
    model=t5_model,
    args=t5_args,
    train_dataset=samsum_tokenized["train"],
    eval_dataset=samsum_tokenized["validation"],
    data_collator=DataCollatorForSeq2Seq(
        tokenizer=t5_tokenizer,
        model=t5_model,
    ),
    compute_metrics=compute_rouge_metrics,
    **trainer_tokenizer_kwargs(Seq2SeqTrainer, t5_tokenizer),
)

t5_trainer.train()
t5_test_output = t5_trainer.predict(samsum_tokenized["test"])
t5_test_metrics = {
    k.replace("test_", ""): float(v)
    for k, v in t5_test_output.metrics.items()
    if isinstance(v, (int, float))
}
RESULTS["t5_summarization"] = t5_test_metrics
display(pd.DataFrame([t5_test_metrics]).round(4))

In [ ]:
@torch.inference_mode()
def summarize_dialogue(dialogue, num_beams=4, max_new_tokens=72):
    t5_model.eval()
    encoded = t5_tokenizer(
        "summarize: " + dialogue,
        return_tensors="pt",
        truncation=True,
        max_length=SOURCE_MAX_LENGTH,
    ).to(t5_model.device)

    output = t5_model.generate(
        **encoded,
        num_beams=num_beams,
        do_sample=False,
        max_new_tokens=max_new_tokens,
    )
    return t5_tokenizer.decode(output[0], skip_special_tokens=True)


generation_rows = []
for idx in range(3):
    sample = samsum["test"][idx]
    generation_rows.append({
        "dialogue": sample["dialogue"],
        "reference": sample["summary"],
        "greedy": summarize_dialogue(sample["dialogue"], num_beams=1),
        "beam_4": summarize_dialogue(sample["dialogue"], num_beams=4),
    })

generation_df = pd.DataFrame(generation_rows)
display(generation_df)

### TODO [SV-D1] — Đánh giá tóm tắt (20 điểm)

1. ROUGE-1, ROUGE-2 và ROUGE-L phản ánh điều gì?
2. So sánh greedy decoding và beam search trên 3 mẫu.
3. Chỉ ra ít nhất một lỗi: bỏ sót ý, thêm thông tin, sai thực thể, lặp, quá dài hoặc quá ngắn.
4. Giải thích vì sao ROUGE cao chưa chắc tóm tắt đã hoàn toàn tốt.
5. Đề xuất một thay đổi về dữ liệu, huấn luyện hoặc decoding.

> **TRẢ LỜI:** ...

In [ ]:
# Giải phóng GPU trước QLoRA.
del t5_trainer, t5_model, t5_test_output
cleanup_gpu()
print(gpu_memory())

## PHẦN E — INSTRUCTION TUNING BẰNG QLoRA (20 điểm)

In [ ]:
LLM_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
ALPACA_DATASET_ID = "bkai-foundation-models/vi-alpaca"

if QUICK_MODE:
    QLORA_TRAIN_SAMPLES = 300
    QLORA_EVAL_SAMPLES = 40
    QLORA_MAX_STEPS = 30
    QLORA_MAX_LENGTH = 256
else:
    QLORA_TRAIN_SAMPLES = 2000
    QLORA_EVAL_SAMPLES = 200
    QLORA_MAX_STEPS = 150
    QLORA_MAX_LENGTH = 384

SYSTEM_PROMPT = (
    "Bạn là trợ lý AI hữu ích. Hãy trả lời bằng tiếng Việt rõ ràng, "
    "đúng trọng tâm và không bịa thông tin."
)

FALLBACK_ALPACA = [
    {
        "instruction": "Giải thích overfitting bằng ngôn ngữ đơn giản.",
        "input": "",
        "output": "Overfitting là khi mô hình ghi nhớ dữ liệu học quá kỹ nên làm tốt trên dữ liệu cũ nhưng kém trên dữ liệu mới.",
    },
    {
        "instruction": "Tóm tắt nội dung sau.",
        "input": "Năng lượng mặt trời là nguồn tái tạo, sạch và ngày càng rẻ hơn.",
        "output": "Năng lượng mặt trời là nguồn năng lượng sạch, tái tạo và ngày càng kinh tế.",
    },
    {
        "instruction": "Nêu ba bước cơ bản để xây dựng mô hình phân loại văn bản.",
        "input": "",
        "output": "Chuẩn bị dữ liệu, huấn luyện mô hình và đánh giá trên tập dữ liệu chưa được dùng để huấn luyện.",
    },
    {
        "instruction": "Phân biệt precision và recall.",
        "input": "",
        "output": "Precision đo tỷ lệ dự đoán dương đúng, còn recall đo tỷ lệ mẫu dương thật được mô hình tìm thấy.",
    },
    {
        "instruction": "Viết lời nhắc học bài ngắn gọn.",
        "input": "",
        "output": "Hãy dành 30 phút tập trung ôn bài hôm nay và tự kiểm tra lại bằng vài câu hỏi ngắn.",
    },
] * 10

try:
    alpaca_raw = load_dataset(ALPACA_DATASET_ID, split="train")
    print("Đã tải:", ALPACA_DATASET_ID)
except Exception as exc:
    print("Không tải được dữ liệu chính:", repr(exc))
    alpaca_raw = Dataset.from_list(FALLBACK_ALPACA)

print(alpaca_raw)
print(alpaca_raw[0])

In [ ]:
def first_text(example, candidates):
    for key in candidates:
        value = example.get(key)
        if value is not None and str(value).strip():
            return str(value).strip()
    return ""


def normalize_alpaca(example):
    instruction = first_text(
        example, ["instruction", "prompt", "question", "query", "user"]
    )
    context = first_text(
        example, ["input", "context", "additional_input", "passage"]
    )
    answer = first_text(
        example, ["output", "response", "answer", "completion", "assistant"]
    )

    user_content = instruction
    if context:
        user_content += f"\n\nThông tin đầu vào:\n{context}"

    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
        ],
        "completion": [
            {"role": "assistant", "content": answer},
        ],
        "instruction_text": instruction,
        "reference_text": answer,
    }


alpaca_normalized = alpaca_raw.map(
    normalize_alpaca,
    remove_columns=alpaca_raw.column_names,
    desc="Chuẩn hóa prompt-completion",
).filter(
    lambda x: bool(x["instruction_text"]) and bool(x["reference_text"]),
    desc="Loại mẫu rỗng",
)

needed = min(
    len(alpaca_normalized),
    QLORA_TRAIN_SAMPLES + QLORA_EVAL_SAMPLES,
)
alpaca_normalized = (
    alpaca_normalized.shuffle(seed=SEED).select(range(needed))
)
eval_size = min(
    QLORA_EVAL_SAMPLES,
    max(1, len(alpaca_normalized) // 10),
)
alpaca_split = alpaca_normalized.train_test_split(
    test_size=eval_size,
    seed=SEED,
)
qlora_train = alpaca_split["train"]
qlora_eval = alpaca_split["test"]

print("Train:", len(qlora_train), "Eval:", len(qlora_eval))
print(qlora_train[0])

In [ ]:
llm_tokenizer = AutoTokenizer.from_pretrained(
    LLM_MODEL_ID,
    use_fast=True,
)
if llm_tokenizer.pad_token is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token
llm_tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=COMPUTE_DTYPE,
    low_cpu_mem_usage=True,
)
llm_model.config.use_cache = False
llm_model = prepare_model_for_kbit_training(
    llm_model,
    use_gradient_checkpointing=True,
)

# TODO [SV-E1]: giải thích r, alpha, dropout và target_modules.
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)

print(count_parameters(llm_model))
print(gpu_memory())

In [ ]:
TEST_PROMPTS = [
    "Giải thích sự khác nhau giữa accuracy và macro-F1.",
    "Nêu ba nguyên nhân khiến mô hình NLP bị overfitting.",
    "Tóm tắt trong hai câu: Transformer dùng cơ chế attention để mô hình hóa quan hệ giữa các token và có thể xử lý song song tốt hơn RNN.",
]

@torch.inference_mode()
def generate_llm_answer(model, prompt, max_new_tokens=120):
    model.eval()
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    text = llm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = llm_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=QLORA_MAX_LENGTH,
    )
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=1.08,
        pad_token_id=llm_tokenizer.pad_token_id,
        eos_token_id=llm_tokenizer.eos_token_id,
    )
    generated = outputs[0, inputs["input_ids"].shape[1]:]
    return llm_tokenizer.decode(
        generated,
        skip_special_tokens=True,
    ).strip()


base_answers = {
    prompt: generate_llm_answer(llm_model, prompt)
    for prompt in TEST_PROMPTS
}
display(pd.DataFrame([
    {"prompt": p, "base_answer": a}
    for p, a in base_answers.items()
]))

In [ ]:
qlora_output_dir = OUTPUT_ROOT / "qwen_qlora_adapter"
if qlora_output_dir.exists():
    shutil.rmtree(qlora_output_dir)

qlora_args = SFTConfig(
    output_dir=str(qlora_output_dir),
    max_steps=QLORA_MAX_STEPS,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    max_grad_norm=0.3,
    optim="paged_adamw_8bit",
    logging_strategy="steps",
    logging_steps=5,
    logging_first_step=True,
    eval_strategy="steps",
    eval_steps=max(10, QLORA_MAX_STEPS // 2),
    save_strategy="no",
    report_to="none",
    seed=SEED,
    data_seed=SEED,
    fp16=not BF16_OK,
    bf16=BF16_OK,
    tf32=USE_TF32,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    use_cache=False,
    max_length=QLORA_MAX_LENGTH,
    packing=False,
    completion_only_loss=True,
    dataset_num_proc=min(2, os.cpu_count() or 1),
    dataloader_num_workers=min(2, os.cpu_count() or 1),
    remove_unused_columns=True,
)

qlora_trainer = SFTTrainer(
    model=llm_model,
    args=qlora_args,
    train_dataset=qlora_train,
    eval_dataset=qlora_eval,
    processing_class=llm_tokenizer,
    peft_config=lora_config,
)

if hasattr(qlora_trainer.model, "print_trainable_parameters"):
    qlora_trainer.model.print_trainable_parameters()

qlora_train_result = qlora_trainer.train()
qlora_eval_metrics = qlora_trainer.evaluate()
qlora_trainer.save_model(str(qlora_output_dir))
llm_tokenizer.save_pretrained(str(qlora_output_dir))

RESULTS["qlora"] = {
    "train_loss": float(
        qlora_train_result.metrics.get("train_loss", np.nan)
    ),
    "eval_loss": float(
        qlora_eval_metrics.get("eval_loss", np.nan)
    ),
    "global_steps": int(qlora_trainer.state.global_step),
    "adapter_path": str(qlora_output_dir),
    "trainable_percent": count_parameters(
        qlora_trainer.model
    )["trainable_percent"],
}

display(pd.DataFrame([RESULTS["qlora"]]))

In [ ]:
after_answers = {
    prompt: generate_llm_answer(qlora_trainer.model, prompt)
    for prompt in TEST_PROMPTS
}

qlora_comparison = pd.DataFrame([
    {
        "prompt": prompt,
        "before_qlora": base_answers[prompt],
        "after_qlora": after_answers[prompt],
    }
    for prompt in TEST_PROMPTS
])
display(qlora_comparison)

### TODO [SV-E2] — Phân tích QLoRA (20 điểm)

1. Giải thích ngắn gọn: LoRA học tham số nào? QLoRA thêm bước gì?
2. Vai trò của `r`, `lora_alpha`, `lora_dropout`, `target_modules`.
3. Tỷ lệ tham số trainable là bao nhiêu? Vì sao nhỏ hơn rất nhiều so với toàn bộ mô hình?
4. So sánh 3 câu trả lời trước/sau QLoRA theo: đúng trọng tâm, rõ ràng, tiếng Việt tự nhiên, có bịa thông tin không.
5. Loss giảm có đủ để kết luận mô hình tốt hơn không? Nêu thêm một cách đánh giá.
6. Nêu một rủi ro khi fine-tune trên dữ liệu chỉ dẫn nhỏ.

> **TRẢ LỜI:** ...

In [ ]:
# Giải phóng GPU sau phần cuối.
del qlora_trainer, llm_model
cleanup_gpu()
print(gpu_memory())

## PHẦN F — BÁO CÁO CUỐI VÀ KIỂM TRA NỘP BÀI (5 điểm)

In [ ]:
summary_rows = [
    {
        "task": "Classification baseline",
        "model": "TF-IDF + Logistic Regression",
        "primary_metric": "Macro-F1",
        "value": RESULTS["baseline_tfidf"]["test_f1_macro"],
    },
    {
        "task": "Classification Transformer",
        "model": "PhoBERT-base-v2",
        "primary_metric": "Macro-F1",
        "value": RESULTS["phobert"].get("f1_macro", np.nan),
    },
    {
        "task": "Summarization",
        "model": "T5-small",
        "primary_metric": "ROUGE-L",
        "value": RESULTS["t5_summarization"].get("rougeL", np.nan),
    },
    {
        "task": "Instruction tuning",
        "model": "Qwen2.5-1.5B + QLoRA",
        "primary_metric": "Eval loss",
        "value": RESULTS["qlora"].get("eval_loss", np.nan),
    },
]

final_summary = pd.DataFrame(summary_rows)
display(final_summary.round(4))

### TODO [SV-F1] — Kết luận cuối (5 điểm)

Viết khoảng 150–250 từ:

- Mô hình nào phù hợp nhất cho từng nhiệm vụ?
- Khi nào baseline truyền thống vẫn là lựa chọn tốt?
- Metric nào quan trọng nhất trong mỗi phần?
- Hạn chế lớn nhất của thí nghiệm là gì?
- Nếu có thêm tài nguyên, bạn sẽ mở rộng bước nào trước?

> **TRẢ LỜI:** ...

In [ ]:
# Checklist kỹ thuật; đây không phải điểm cuối cùng.
checks = {
    "GPU available": torch.cuda.is_available(),
    "Baseline result exists": "baseline_tfidf" in RESULTS,
    "PhoBERT result exists": "phobert" in RESULTS,
    "T5 result exists": "t5_summarization" in RESULTS,
    "QLoRA result exists": "qlora" in RESULTS,
    "QLoRA adapter saved": Path(
        RESULTS.get("qlora", {}).get("adapter_path", "")
    ).exists(),
}

check_df = pd.DataFrame({
    "check": checks.keys(),
    "passed": checks.values(),
})
display(check_df)

assert all(checks.values()), (
    "Còn phần chưa chạy thành công. Kiểm tra lại các cell phía trên."
)
print("Checklist kỹ thuật đã đạt.")

In [ ]:
def to_jsonable(value):
    if isinstance(value, dict):
        return {str(k): to_jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_jsonable(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, Path):
        return str(value)
    return value


# TODO [SV]: điền thông tin trước khi xuất báo cáo.
report = {
    "student": {
        "name": "TODO",
        "student_id": "TODO",
        "class": "TODO",
    },
    "environment": {
        "gpu": GPU_NAME,
        "vram_gb": GPU_VRAM_GB,
        "quick_mode": QUICK_MODE,
        "seed": SEED,
    },
    "results": RESULTS,
}

report_path = Path("/content/lab12_final_report.json")
with report_path.open("w", encoding="utf-8") as f:
    json.dump(
        to_jsonable(report),
        f,
        ensure_ascii=False,
        indent=2,
    )

print("Đã tạo:", report_path)
print("Hãy tải file JSON và notebook .ipynb để nộp.")

## Gợi ý chấm dành cho giảng viên

### Phần A — 5 điểm
- Chạy đúng T4, seed cố định, giải thích FP16 và quản lý VRAM.

### Phần B — 20 điểm
- EDA đúng và có nhận xét: 5.
- Pipeline baseline đúng, không rò rỉ dữ liệu: 7.
- Metric, confusion matrix, phân tích lỗi và lý do chọn siêu tham số: 8.

### Phần C — 30 điểm
- Chuẩn bị dữ liệu/tokenizer đúng: 7.
- Fine-tune và đánh giá đúng tập test: 10.
- So sánh baseline, phân tích lỗi có dẫn chứng: 9.
- Đề xuất cải tiến khả thi trên T4: 4.

### Phần D — 20 điểm
- Pipeline seq2seq đúng: 8.
- ROUGE và so sánh decoding: 6.
- Phân tích lỗi và hạn chế của metric: 6.

### Phần E — 20 điểm
- Hiểu LoRA/QLoRA và cấu hình adapter: 7.
- Chạy được QLoRA, lưu adapter, báo cáo trainable parameters: 7.
- So sánh định tính trước/sau và nêu rủi ro: 6.

### Phần F — 5 điểm
- Kết luận logic, nộp đủ notebook và JSON, output không bị xóa.

> Không nên chấm cứng theo một ngưỡng metric duy nhất vì kết quả có thể dao động theo runtime, phiên bản thư viện và mẫu dữ liệu được lấy.

## Câu hỏi vấn đáp ngắn tùy chọn

1. Vì sao Macro-F1 phù hợp hơn Accuracy khi lớp mất cân bằng?
2. Vì sao không dùng tập test để chọn hyperparameter?
3. Attention khác RNN ở khả năng xử lý song song như thế nào?
4. Beam search có luôn tốt hơn greedy decoding không?
5. Vì sao 4-bit giúp QLoRA tiết kiệm VRAM?
6. Adapter LoRA có thể được dùng lại như thế nào?